**Figure 5: TRH-D (base) vs Lookback Window L, Fixed W=72**

In [ ]:
# ============================================================================
# Figure 5: TRH-D (base) vs lookback window L, fixed W=72
# ============================================================================
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter, NullFormatter, NullLocator

# ---- Config (edit these freely in the notebook) ----
# Adjust if your notebook lives somewhere other than security_analysis/
RESULTS_DIR = Path("../results")
OUTPUT_DIR  = Path("../plots")
W       = 72
QTH     = 4
ABO_ACT = 12
# ----------------------------------------------------

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 11
sns.set_palette("tab10")
sns.set_style("whitegrid")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tag = f"QTH{QTH}_ABO{ABO_ACT}"
csv_path = RESULTS_DIR / f"figure8_generalized_with_mc_W{W}_{tag}.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find {csv_path}. Run the 'figure8' subcommand first.")
df = pd.read_csv(csv_path)

# Plot the *base* TRH-D (before the QTH + ABO_ACT margin) so the curve
# reflects only the analytical security contribution.
analytic_col = "TRH_D_base_analytical"
sim_col = "TRH_D_base_simulated"
if analytic_col not in df.columns:
    raise ValueError(f"Missing required column: {analytic_col}")
has_mc = sim_col in df.columns

r_values = sorted(df["R"].dropna().unique())
l_min, l_max = df["L"].min(), df["L"].max()

fig, ax = plt.subplots(figsize=(4.5, 1.8))
colors = list(plt.get_cmap("tab10").colors)
handles = []

for idx, R in enumerate(r_values):
    sub = df[df["R"] == R].sort_values("L")
    if sub.empty:
        continue
    color = colors[idx % len(colors)]

    ax.plot(sub["L"], sub[analytic_col], color=color, linewidth=1.6, zorder=1)
    handles.append(mlines.Line2D([], [], color=color, linewidth=1.6,
                                 label=f"R = {int(R)}"))

    if has_mc:
        mc_sub = sub.dropna(subset=[sim_col])
        if not mc_sub.empty:
            ax.plot(mc_sub["L"], mc_sub[sim_col],
                    marker="o", markersize=4.8, markerfacecolor="none",
                    markeredgecolor=color, markeredgewidth=0.7,
                    linestyle="None", zorder=2)

ax.set_xlabel("Lookback Window (L)", fontsize=11)
ax.set_ylabel("Double-Sided RowHammer\nThreshold (T$_{RH-D}$)", fontsize=11)
ax.tick_params(axis="both", which="major", labelsize=11)

ax.set_yticks([500, 750, 1000, 1250])
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y)}"))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_minor_locator(NullLocator())
ax.tick_params(axis="y", which="minor", left=False)
ax.set_ylim(350, 1400)
ax.set_xlim(l_min - 0.5, l_max + 0.5)
ax.grid(True, linewidth=0.7, alpha=0.7)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("0.7")

top_legend = ax.legend(handles=handles, loc="upper center",
                       bbox_to_anchor=(0.5, 1.55),
                       title="Sampled Activation Slots (R)",
                       ncol=4, fontsize=9, frameon=True)
ax.add_artist(top_legend)

if has_mc:
    ax.legend(handles=[mlines.Line2D([], [], color="black", marker="o",
                                     markerfacecolor="none",
                                     markeredgewidth=0.9,
                                     linestyle="None", label="Simulation")],
              loc="best", fontsize=9.5, frameon=True, framealpha=0.9)

fig_path = OUTPUT_DIR / f"sens_R_L_TRHD_W{W}_{tag}.pdf"
fig.savefig(fig_path, dpi=600, bbox_inches="tight",
            bbox_extra_artists=(top_legend,), pad_inches=0.05)
print(f"Saved: {fig_path}")
plt.show()  # display inline in the notebook

**Figure 6: Pareto frontier of TRH-D (base) vs SHQ entries, swept across W**

In [ ]:
# ============================================================================
# Figure 6: Pareto frontier of TRH-D (base) vs SHQ entries, swept across W.
# ============================================================================
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FixedLocator, FuncFormatter

# ---- Config (edit these freely in the notebook) ----
RESULTS_DIR        = Path("../results")
OUTPUT_DIR         = Path("../plots")
QTH                = 4
ABO_ACT            = 12
W_VALUES           = [72, 60, 48, 36, 24]
MAX_SHQ_ENTRIES    = 300   # truncate uninteresting storage tail
SAVE_FRONTIER_CSV  = False
# ----------------------------------------------------

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 11
sns.set_palette("tab10")
sns.set_style("whitegrid")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tag = f"QTH{QTH}_ABO{ABO_ACT}"
csv_path = RESULTS_DIR / f"sweep_W_R_L_{tag}.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find {csv_path}. Run the 'sweep-w' subcommand first.")
df = pd.read_csv(csv_path)

required = {"W", "R", "L", "SHQ_entries", "TRH_D_base"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[df["W"].isin(W_VALUES)].copy()

# (R=2, L=2) collapses PrISM to a MINT-equivalent design point. Exclude it
# so the frontier reflects only true PrISM configurations.
df = df[~((df["R"] == 2) & (df["L"] == 2))].copy()


def build_frontier(df_w: pd.DataFrame) -> pd.DataFrame:
    """Pareto frontier on (SHQ_entries, TRH_D_base): keep only points where
    increasing SHQ_entries strictly improves (lowers) TRH_D_base."""
    grouped = (
        df_w.sort_values(["SHQ_entries", "TRH_D_base", "R", "L"])
            .groupby("SHQ_entries", as_index=False)
            .first()
            .sort_values("SHQ_entries")
            .reset_index(drop=True)
    )
    frontier = []
    best_so_far = float("inf")
    for _, row in grouped.iterrows():
        if row["TRH_D_base"] < best_so_far:
            frontier.append(row)
            best_so_far = row["TRH_D_base"]
    return (pd.DataFrame(frontier).reset_index(drop=True)
            if frontier else pd.DataFrame(columns=grouped.columns))


frontiers = {}
for W in W_VALUES:
    sub = df[df["W"] == W]
    if sub.empty:
        continue
    f = build_frontier(sub)
    f = f[f["SHQ_entries"] <= MAX_SHQ_ENTRIES]
    frontiers[W] = f

if SAVE_FRONTIER_CSV:
    rows = []
    preferred_cols = [
        "W", "R", "L", "SHQ_entries", "TRH_D_base", "TRH_D", "TRH_D_margin",
        "A_star", "X_win_star", "N_per_row_star", "p_m_star",
        "sampling_fraction", "default_mitigation_rate",
    ]
    for W, f in frontiers.items():
        if f.empty:
            continue
        f = f.copy()
        f["frontier_used_in_plot"] = True
        rows.append(f)
    if rows:
        df_frontier = pd.concat(rows, ignore_index=True)
        cols = [c for c in preferred_cols if c in df_frontier.columns]
        extras = [c for c in df_frontier.columns if c not in cols]
        df_frontier = df_frontier[cols + extras]
        frontier_csv = OUTPUT_DIR / f"frontier_W_SHQ_TRHDbase_leq{MAX_SHQ_ENTRIES}_{tag}.csv"
        df_frontier.to_csv(frontier_csv, index=False)
        print(f"Saved frontier points: {frontier_csv}")

fig, ax = plt.subplots(figsize=(4.5, 1.8))
colors = list(plt.get_cmap("tab10").colors)
handles = []

for idx, W in enumerate(W_VALUES):
    f = frontiers.get(W)
    if f is None or f.empty:
        continue
    color = colors[idx % len(colors)]
    ax.plot(f["SHQ_entries"], f["TRH_D_base"],
            color=color, linewidth=1.7, marker="o",
            markersize=3, zorder=1)
    handles.append(mlines.Line2D([], [], color=color, linewidth=1.7,
                                 marker="o", markersize=3, label=f"W = {W}"))

ax.set_xlabel("Sampled History Queue (SHQ) Entries", fontsize=11)
ax.set_ylabel("Double-Sided RowHammer\nThreshold (T$_{RH-D}$)", fontsize=11)
ax.tick_params(axis="both", which="major", labelsize=10.5)

yticks = [125, 250, 500, 750, 1000]
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y)}"))
ax.set_ylim(150, 1100)
ax.set_xlim(-5, MAX_SHQ_ENTRIES + 10)
ax.grid(True, linewidth=0.7, alpha=0.7)

top_legend = ax.legend(handles=handles, loc="upper center",
                       bbox_to_anchor=(0.5, 1.55),
                       title="Mitigation Window Size (W)",
                       ncol=3, fontsize=10, frameon=True)
ax.add_artist(top_legend)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_color("0.7")

fig_path = OUTPUT_DIR / f"sens_W_SHQ_TRHDbase_{tag}.pdf"
fig.savefig(fig_path, dpi=600, bbox_inches="tight",
            bbox_extra_artists=(top_legend,), pad_inches=0.05)
print(f"Saved: {fig_path}")
plt.show()  # display inline in the notebook